# Module 3.5 - Evaluating and Optimising Retrieval

**Reference:** [`05-evaluating-and-optimising-retrieval.md`](./05-evaluating-and-optimising-retrieval.md)

## What you'll do in this notebook

- Build a tiny labelled test set - (query, relevant-doc-ids) pairs.
- Implement **Precision@K**, **Recall@K**, and **MRR** from scratch.
- Use these metrics to compare two retrieval strategies and decide which one to ship.

## Setup

We'll set up two toy retrievers so you can compare them. One is plain vector search; the other adds query expansion (from section 3.1).

In [ ]:
import os
import shutil
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBED_MODEL = "text-embedding-3-small"
CHROMA_PATH = "./chroma_store_m3_5"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma.create_collection("eval_demo")

def embed_batch(texts):
    return [item.embedding for item in client.embeddings.create(model=EMBED_MODEL, input=texts).data]

DOCS = {
    "py_errors":     "Exception handling in Python uses try, except, and finally blocks.",
    "py_context":    "Python context managers free resources deterministically via __enter__ and __exit__.",
    "py_listcomp":   "A Python list comprehension concisely builds a list from an iterable.",
    "py_decorator":  "A Python decorator wraps a function to extend its behaviour.",
    "js_errors":     "JavaScript error handling uses try, catch, and finally blocks.",
    "js_async":      "Async/await in JavaScript was standardised in ES2017.",
    "react_intro":   "React is a JavaScript library by Meta for building user interfaces.",
    "django_intro":  "Django is a high-level Python web framework.",
}

ids = list(DOCS.keys())
texts = list(DOCS.values())
collection.add(ids=ids, embeddings=embed_batch(texts), documents=texts, metadatas=[{} for _ in ids])
print(f"OK: {collection.count()} docs indexed")

## Exercise 1 - A hand-built test set

Each test case pairs a query with the set of doc IDs that *should* be retrieved for it. This is your ground truth.

In [ ]:
TEST_SET: list[dict] = [
    {"query": "How do I handle exceptions in Python?",      "relevant": {"py_errors"}},
    {"query": "catch errors in Python code",                "relevant": {"py_errors"}},
    {"query": "how do list comprehensions work",            "relevant": {"py_listcomp"}},
    {"query": "what are context managers for",              "relevant": {"py_context"}},
    {"query": "JavaScript error handling",                  "relevant": {"js_errors"}},
    {"query": "What is React used for?",                    "relevant": {"react_intro"}},
    {"query": "Python web frameworks",                      "relevant": {"django_intro"}},
    {"query": "async functions in JavaScript",              "relevant": {"js_async"}},
]
# NOTE: a "good" test set has at least 20-30 queries and mixes easy and hard cases.
# Eight is enough for this exercise but too few for a real evaluation.

## Exercise 2 - Implement the metrics

In [ ]:
def retrieve_basic(query: str, top_k: int = 5) -> list[str]:
    """Plain vector search - returns a list of doc IDs in rank order."""
    emb = embed_batch([query])[0]
    res = collection.query(query_embeddings=[emb], n_results=top_k)
    return res["ids"][0]

def precision_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    # TODO: fraction of the top-k retrieved ids that are in `relevant`.
    raise NotImplementedError

def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    # TODO: fraction of `relevant` that appear in the top-k retrieved ids.
    raise NotImplementedError

def reciprocal_rank(retrieved: list[str], relevant: set[str]) -> float:
    # TODO: 1 / (rank of first relevant id) with rank starting at 1;
    # return 0.0 if no relevant id appears.
    raise NotImplementedError

# Quick sanity check on one query
first = TEST_SET[0]
hits = retrieve_basic(first["query"], top_k=5)
print(f"retrieved: {hits}")
print(f"P@5  = {precision_at_k(hits, first['relevant'], k=5):.3f}")
print(f"R@5  = {recall_at_k(hits, first['relevant'], k=5):.3f}")
print(f"MRR  = {reciprocal_rank(hits, first['relevant']):.3f}")

## Exercise 3 - A second strategy: query expansion

Recreate the basic retrieval from 3.1 so we have two strategies to compare.

In [ ]:
def expand(query: str, n: int = 3) -> list[str]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Return n alternative phrasings of the user question, one per line, no numbering."},
            {"role": "user", "content": f"Question: {query}\nN: {n}"},
        ],
        temperature=0.4,
    )
    lines = [line.strip(" -*\t") for line in resp.choices[0].message.content.splitlines() if line.strip()]
    return [query] + lines[:n]

def retrieve_expanded(query: str, top_k: int = 5) -> list[str]:
    """Vector search over each expansion, then pool + frequency-rank."""
    # TODO:
    # 1. variants = expand(query, n=3)
    # 2. For each variant, call retrieve_basic(variant, top_k=top_k) and collect ids.
    # 3. Deduplicate while counting how often each id appears.
    # 4. Sort ids desc by count (ties broken by first appearance) and return top_k.
    raise NotImplementedError


## Exercise 4 - Compare the two strategies

Run every query through both strategies, average the three metrics, and print a comparison row.

In [ ]:
def evaluate(retrieve_fn, test_set: list[dict], k: int = 5) -> dict:
    # TODO: loop over test_set, compute P@k, R@k, MRR for each case,
    # and return {"p_at_k": mean_p, "r_at_k": mean_r, "mrr": mean_mrr}.
    raise NotImplementedError

print(f"{'strategy':<18} {'P@5':>6} {'R@5':>6} {'MRR':>6}")
for name, fn in [("basic", retrieve_basic), ("expand", retrieve_expanded)]:
    metrics = evaluate(fn, TEST_SET, k=5)
    print(f"{name:<18} {metrics['p_at_k']:>6.3f} {metrics['r_at_k']:>6.3f} {metrics['mrr']:>6.3f}")

## Reflection

- Which strategy wins overall? Is the margin large enough to justify the extra LLM and embedding calls query expansion requires?
- If you flipped between `retrieve_basic` and `retrieve_reranked` (section 3.2), how would you predict the metrics to move?
- What's the cheapest single change you could make to this test set to get stronger signal: more queries, more diverse wording, or more relevant docs per query?

## Capstone checkpoint 3

- Write 15-20 (query, relevant_doc_ids) pairs for *your* capstone corpus.
- Run `evaluate` on your capstone's retrieval, then on a version with at least one Module 3 improvement (expansion, reranking, hybrid, or caching).
- Record the numbers in your project's README. These are the baselines you'll report in the Module 8 capstone results table.

## What's next

Module 4 moves from *what the bot knows* to *what the bot remembers*. You'll handle growing conversation histories with summarisation, selective memory, and retrieval-based recall. Start with `module_4/01-conversation-history-management.ipynb`.